In [ ]:
pip install ptflops

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler
from ptflops import get_model_complexity_info
import time

In [ ]:
BATCH_SIZE = 16
LR = 0.001
EPOCHS = 5
NUM_CLASSES = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [ ]:
def get_dataloaders(batch_size):
    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ])

    train_ds = torchvision.datasets.FashionMNIST(
        root="./data", train=True, download=True, transform=transform
    )

    test_ds = torchvision.datasets.FashionMNIST(
        root="./data", train=False, download=True, transform=transform
    )

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=2
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=2
    )

    return train_loader, test_loader


In [ ]:
def build_model(name):
    if name == "resnet18":
        model = torchvision.models.resnet18(weights=None)
    elif name == "resnet32":
        model = torchvision.models.resnet34(weights=None)  # used as ResNet-32
    elif name == "resnet50":
        model = torchvision.models.resnet50(weights=None)

    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(DEVICE)


In [ ]:
def get_optimizer(model, optimizer_name, lr):
    if optimizer_name == "SGD":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_name == "Adam":
        return optim.Adam(model.parameters(), lr=lr)


In [ ]:
def compute_flops(model):
    model.eval()

    with torch.no_grad():
        flops, params = get_model_complexity_info(
            model,
            (3, 224, 224),
            as_strings=False,
            print_per_layer_stat=False,
            verbose=False
        )

    return flops


In [ ]:
def train_and_test(model, optimizer, train_loader, test_loader):
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

    start_time = time.time()

    for _ in range(EPOCHS):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()

            with autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(x)
                loss = criterion(outputs, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    train_time_ms = (time.time() - start_time) * 1000

    # ---------- Testing ----------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    accuracy = 100.0 * correct / total
    return accuracy, train_time_ms


In [ ]:
train_loader, test_loader = get_dataloaders(BATCH_SIZE)

experiments = [
    ("resnet18", "SGD"),
    ("resnet18", "Adam"),
    ("resnet32", "SGD"),
    ("resnet32", "Adam"),
    ("resnet50", "SGD"),
    ("resnet50", "Adam"),
]

print("\nModel     | Optimizer | Accuracy (%) | Train Time (ms) | FLOPs")
print("-" * 95)

for model_name, opt_name in experiments:
    model = build_model(model_name)

    # ---- FLOPs (once per model) ----
    flops = compute_flops(model)

    optimizer = get_optimizer(model, opt_name, LR)

    acc, time_ms = train_and_test(
        model, optimizer, train_loader, test_loader
    )

    print(f"{model_name:<9} | {opt_name:<9} | "
          f"{acc:12.2f} | {time_ms:14.2f} | {flops/1e9:6.2f} GFLOPs")


100%|██████████| 26.4M/26.4M [00:01<00:00, 17.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 274kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.54MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.6MB/s]



Model     | Optimizer | Accuracy (%) | Train Time (ms) | FLOPs
-----------------------------------------------------------------------------------------------


/tmp/ipykernel_55/483473401.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE.type == "cuda"))
/tmp/ipykernel_55/483473401.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE.type == "cuda")):


resnet18  | SGD       |        91.36 |      557896.22 |   1.82 GFLOPs
resnet18  | Adam      |        92.92 |      576908.84 |   1.82 GFLOPs
resnet32  | SGD       |        92.51 |      860304.00 |   3.68 GFLOPs
resnet32  | Adam      |        90.44 |      926024.53 |   3.68 GFLOPs
resnet50  | SGD       |        90.12 |     1315006.90 |   4.13 GFLOPs
resnet50  | Adam      |        90.93 |     1389783.36 |   4.13 GFLOPs


In [ ]:
import torch

# Path where the model will be saved
MODEL_PATH = "Q2_GPU.pth"

# Save only model parameters (recommended)
torch.save(model.state_dict(), MODEL_PATH)

print(f"Model saved to {MODEL_PATH}")


CPU

In [ ]:
pip install ptflops

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler
from ptflops import get_model_complexity_info
import time
import warnings
warnings.filterwarnings("ignore")

In [ ]:
BATCH_SIZE = 16
LR = 0.001
EPOCHS = 1
NUM_CLASSES = 10

#DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

Using device: cpu


In [ ]:
def get_dataloaders(batch_size):
    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ])

    train_ds = torchvision.datasets.FashionMNIST(
        root="./data", train=True, download=True, transform=transform
    )

    test_ds = torchvision.datasets.FashionMNIST(
        root="./data", train=False, download=True, transform=transform
    )

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=2
    )

    test_loader = torch.utils.data.DataLoader(
        test_ds, batch_size=batch_size, shuffle=False, num_workers=2
    )

    return train_loader, test_loader


In [ ]:
def build_model(name):
    if name == "resnet18":
        model = torchvision.models.resnet18(weights=None)
    elif name == "resnet32":
        model = torchvision.models.resnet34(weights=None)  # used as ResNet-32
    elif name == "resnet50":
        model = torchvision.models.resnet50(weights=None)

    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(DEVICE)


In [ ]:
def get_optimizer(model, optimizer_name, lr):
    if optimizer_name == "SGD":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_name == "Adam":
        return optim.Adam(model.parameters(), lr=lr)


In [ ]:
def compute_flops(model):
    model.eval()

    with torch.no_grad():
        flops, params = get_model_complexity_info(
            model,
            (3, 224, 224),
            as_strings=False,
            print_per_layer_stat=False,
            verbose=False
        )

    return flops


In [ ]:
def train_and_test(model, optimizer, train_loader, test_loader):
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

    start_time = time.time()

    for _ in range(EPOCHS):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()

            with autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(x)
                loss = criterion(outputs, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    train_time_ms = (time.time() - start_time) * 1000

    # ---------- Testing ----------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    accuracy = 100.0 * correct / total
    return accuracy, train_time_ms


In [ ]:
train_loader, test_loader = get_dataloaders(BATCH_SIZE)

experiments = [
    ("resnet18", "SGD"),
    ("resnet18", "Adam"),
    ("resnet50", "SGD"),
    ("resnet50", "Adam"),
]

print("\nModel     | Optimizer | Test Classification Accuracy (%) | Train Time (ms) | FLOPs")
print("-" * 95)

for model_name, opt_name in experiments:
    model = build_model(model_name)

    # ---- FLOPs (once per model) ----
    flops = compute_flops(model)

    optimizer = get_optimizer(model, opt_name, LR)

    acc, time_ms = train_and_test(
        model, optimizer, train_loader, test_loader
    )

    print(f"{model_name:<9} | {opt_name:<9} | "
          f"{acc:12.2f} | {time_ms:14.2f} | {flops/1e9:6.2f} GFLOPs")


Model     | Optimizer | Test Classification Accuracy (%) | Train Time (ms) | FLOPs
-----------------------------------------------------------------------------------------------
resnet18  | SGD       |        87.45 |    15899162.25 |   1.82 GFLOPs
